In [21]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field
from typing import List, Optional
from langchain_openai import ChatOpenAI
from langchain import hub
from uuid import uuid4
from pathlib import Path
from langgraph.checkpoint.memory import MemorySaver
import re
from PIL import Image
import uuid, json, base64, os, io
import pandas as pd

### Original Function and Class

In [8]:
class FoodItem(BaseModel):
    food_item: str = Field(..., description="Name of the identified food")
    portion: str = Field(..., description="Estimated portion size, e.g., 1 cup, 150g")
    calories: int = Field(..., description="Estimated calorie content")
    protein_g: float = Field(..., description="Estimated protein in grams")
    fat_g: float = Field(..., description="Estimated total fat in grams")
    saturated_fat_g: float = Field(..., description="Estimated saturated fat in grams")
    carbs_g: float = Field(..., description="Estimated carbohydrate content in grams")


class NutritionEstimate(BaseModel):
    items: List[FoodItem]
    total: FoodItem


prompt = """
You are a food nutrition expert.

Estimate the nutrition of the meal in the provided image.

1. Identify visible food items.
2. Estimate portion size for each.
3. Provide the following for each item:
   - Calories (kcal)
   - Protein (g)
   - Fat (g)
   - Saturated Fat (g)
   - Carbohydrates (g)
4. Provide a total combined nutritional summary for the entire meal.

If any assumptions are made (e.g., portion or food type), estimate them reasonably.
"""

header_name_mapping = {'food_item': 'Food Item', 'portion': 'Portion', 'calories': 'Calories',
                       'protein_g': 'Protein(g)', 'fat_g': 'Fat(g)',
                       'saturated_fat_g': 'Sat. Fat(g)', 'carbs_g': 'Carbs. (g)',
                       'meal_date': 'Meal Date', 'meal_type': 'Meal Type'
                       }

In [9]:
def base64_to_image(base64_str):
    match = re.match(r'^data:image/(?P<fmt>\w+);base64,', base64_str)
    if not match:
        raise ValueError("Unsupported base64 image format (missing data:image/xxx;base64, prefix)")

    img_format = match.group("fmt").upper()
    base64_data = re.sub('^data:image/.+;base64,', '', base64_str)
    byte_data = base64.b64decode(base64_data)
    image_data = io.BytesIO(byte_data)
    img = Image.open(image_data)

    return img, img_format


def image_to_base64(img, img_format="JPEG"):
    buffer = io.BytesIO()

    # Pillow uses "JPEG", not "JPG"
    if img_format == "JPG" or img_format.lower() == "heic":
        img_format = "JPEG"

    img.save(buffer, format=img_format)
    base64_str = base64.b64encode(buffer.getvalue()).decode()
    return f"data:image/{img_format.lower()};base64,{base64_str}"


def process_upload_img(contents: str, b_crop_to_512: bool = False):
    img, img_format = base64_to_image(contents)

    if b_crop_to_512:
        # Crop to center square
        width, height = img.size
        min_dim = min(width, height)
        left = (width - min_dim) // 2
        top = (height - min_dim) // 2
        right = left + min_dim
        bottom = top + min_dim
        img_cropped = img.crop((left, top, right, bottom))
        img_resized = img_cropped.resize((512, 512), Image.LANCZOS)
    else:
        # Resize with aspect ratio preserved, long side = 512
        width, height = img.size
        if width >= height:
            new_width = 512
            new_height = int(height * 512 / width)
        else:
            new_height = 512
            new_width = int(width * 512 / height)
        img_resized = img.resize((new_width, new_height), Image.LANCZOS)

    if img_format.lower() == "heic":
        img_format = "JPEG"

    base64_resized_str = image_to_base64(img_resized, img_format)

    return img_resized, base64_resized_str


def parse_response_to_dataframe(response):
    df = pd.DataFrame([item.model_dump() for item in response.items])
    total_row = pd.DataFrame([response.total.model_dump()])
    df_with_total = pd.concat([df, total_row], ignore_index=True)

    return df_with_total

In [10]:
llm = ChatOpenAI(
    model="gpt-4o-mini",  # or "gpt-4o-mini" if available in your org
    temperature=0,
    api_key=os.environ.get('OPENAI_API_KEY')
)

llm_with_output = llm.with_structured_output(NutritionEstimate)

### Graph Build

In [ ]:
class State(BaseModel):
    base64_image: str
    feedback: Optional[str] = None
    llm_output: Optional[NutritionEstimate] = None  # 已解析后的 Pydantic 对象
    feedback_history: List[str] = []


def init_recognition_node(state: State):
    b64_img = state.base64_image
    response = llm_with_output.invoke([
        {"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": b64_img}}
        ]}
    ])
    return {'llm_output': response, 'feedback_history': []}


def user_feedback_node(state: State):
    b64_img = state.base64_image
    feedback = state.feedback
    prev_result = state.llm_output

    fb_prompt = f"""
    You are a food recognition expert.

    This is the previous recognition output:
    {prev_result.model_dump_json(indent=2)}

    The user provided the following correction:
    "{feedback}"

    Please re-evaluate the image and update the nutrition estimates accordingly.
    """
    res = llm_with_output.invoke([
        {"role": "user", "content": [
            {"type": "text", "text": fb_prompt},
            {"type": "image_url", "image_url": {"url": b64_img}}
        ]}
    ])
    hist = state.feedback_history + [feedback]

    return {"llm_output": res, "feedback_history": hist}


def build_graph():
    sg = StateGraph(State)
    sg.add_node("init_recognition", init_recognition_node)
    sg.add_node("user_feedback", user_feedback_node)
    # 流程: initial → feedback (可循环) → END
    sg.set_entry_point("init_recognition")
    sg.add_edge("init_recognition", "user_feedback")
    sg.add_conditional_edges(
        "user_feedback",
        lambda s: END if len(s.feedback_history) >= 3 else "user_feedback"
    )
    saver = MemorySaver()  # 本地持久化
    return sg.compile(checkpointer=saver)


graph = build_graph()


In [31]:


### TEST ####
input_path = r"E:\Projects\Selfwork\FoodLens\app\data\mixed_rice.jpg" # Use your own test image path

# Step 1: Open and encode test image to base64 with data URI prefix
with open(input_path, "rb") as image_file:
    base64_encoded = base64.b64encode(image_file.read()).decode()
    contents = f"data:image/jpeg;base64,{base64_encoded}"

# Load Image
img_processed, base64_out = process_upload_img(contents)
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

# Input image to trigger the graph
state = {"base64_image": base64_out}
graph.invoke(state, config=config, interrupt_before=['user_feedback'])  # 跑到 user_feedback 前停

# 拿最新结果
latest = graph.get_state(config).values["llm_output"]
parsed_df = parse_response_to_dataframe(latest)

In [32]:
config = {"configurable": {"thread_id": thread_id}}
# 把用户反馈写入 state，再往前推进一步
graph.update_state(config, {"feedback": 'It is not fried chicken, it is beef'})
graph.invoke({}, config=config)  # 执行 user_feedback 节点

latest_fb = graph.get_state(config).values["llm_output"]
parsed_fb_df = parse_response_to_dataframe(latest_fb)

In [33]:
parsed_fb_df

,food_item,portion,calories,protein_g,fat_g,saturated_fat_g,carbs_g
0,Beef (Grilled or Stir-Fried),3 pieces (150g),350,30.0,20.0,8.0,0.0
1,Steamed White Rice,1 cup (150g),205,4.0,0.4,0.1,45.0
2,"Stir-Fried Vegetables (Broccoli, Carrots, Bell...",1 cup (150g),70,3.0,0.5,0.1,15.0
3,Stir-Fried Purple Eggplant,1 cup (150g),120,2.0,7.0,1.0,15.0
4,Total Meal,N/A,845,39.0,28.9,9.2,75.0


In [1]:
from openai import OpenAI
import os
client = OpenAI(
    api_key = os.getenv('OPENAI_API_KEY')
)
models = client.models.list()
for model in models:
    print(model.id)

gpt-4o-audio-preview-2024-12-17
dall-e-3
dall-e-2
gpt-4o-audio-preview-2024-10-01
gpt-4-turbo-preview
text-embedding-3-small
gpt-4-1106-preview
gpt-4-turbo
gpt-4-turbo-2024-04-09
gpt-4.1-nano
gpt-4.1-nano-2025-04-14
gpt-4o-realtime-preview-2024-10-01
gpt-4o-realtime-preview
babbage-002
gpt-4
text-embedding-ada-002
chatgpt-4o-latest
gpt-4o-mini-audio-preview
gpt-4o-audio-preview
o1-preview-2024-09-12
gpt-4o-mini-realtime-preview
gpt-4.1-mini
gpt-4o-mini-realtime-preview-2024-12-17
gpt-3.5-turbo-instruct-0914
gpt-4o-mini-search-preview
gpt-4.1-mini-2025-04-14
o1
gpt-3.5-turbo-16k
o1-2024-12-17
davinci-002
gpt-3.5-turbo-1106
gpt-4o-search-preview
gpt-3.5-turbo-instruct
gpt-3.5-turbo
gpt-4o-mini-search-preview-2025-03-11
gpt-4-0125-preview
gpt-4o-2024-11-20
whisper-1
gpt-4o-2024-05-13
o1-pro
o1-pro-2025-03-19
o1-preview
gpt-4-0613
gpt-image-1
text-embedding-3-large
gpt-4o-mini-tts
gpt-4o-transcribe
gpt-4.5-preview
gpt-4.5-preview-2025-02-27
gpt-4o-mini-transcribe
gpt-4o-search-preview-2025